# A pie · Percepción y accidentalidad peatonal en Madrid

Este cuaderno conecta tu archivo **accidentes_madrid.duckdb** con la plataforma de comparaciones. Es autónomo: las funciones están incluidas en el propio cuaderno.

1. Leer y auditar la base de accidentes sin modificarla.
2. Contar accidentes con peatón en un radio común alrededor de las dos ubicaciones.
3. Preparar el mapa de Madrid, los focos de acumulación y un catálogo importable.
4. Añadir tablas `pc_*` a DuckDB si activas esa opción.
5. Analizar las respuestas descargadas desde la vista privada.

**No se han ejecutado los cálculos sobre tu archivo de Windows:** los resultados aparecerán al ejecutarlo allí. No hay recuentos inventados. La primera ubicación es una hipótesis de mayor accidentalidad y la segunda una candidata de contraste; sus etiquetas se verifican con los registros.

El piloto repite una pareja cinco veces. No sirve para estimar diferencias entre calles de todo Madrid. El análisis principal excluye automáticamente las sesiones de este piloto.


## 1. Dependencias
Ejecuta esta celda una vez. Si Jupyter solicita reiniciar el kernel, reinícialo antes de continuar. El backend portable tiene sus dependencias en `requirements.txt`, pero no es necesario arrancarlo para analizar el archivo.

In [ ]:
%pip install "duckdb>=1.2,<2" "pandas>=2.1,<4" "numpy>=1.26,<3" "pyproj>=3.6,<4" "scipy>=1.12,<2" "scikit-learn>=1.4,<2" "statsmodels>=0.14,<1" "matplotlib>=3.8,<4"

## 2. Funciones del estudio
Ejecuta esta celda completa. Incluye los cálculos espaciales, las tablas nuevas y el análisis. Las distancias se miden en **EPSG:25830**, nunca en grados de latitud/longitud. No utiliza DuckDB Spatial ni necesita descargar extensiones.

In [ ]:
"""Preparación espacial y análisis reproducible del estudio A pie.

No requiere DuckDB Spatial: mide en EPSG:25830 usando pyproj y cKDTree.
Los accidentes se cuentan en accidentes_todos, sin uniones a personas_todos.
"""
from __future__ import annotations
import hashlib
import json
import math
from collections import defaultdict
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from pyproj import Transformer
from scipy.spatial import cKDTree

DEFINITION='unique_accident_with_pedestrian_police_record'
QUALITY='high|medium;apto_mapa=true'
TO_UTM=Transformer.from_crs(4326,25830,always_xy=True)
TO_WGS=Transformer.from_crs(25830,4326,always_xy=True)
DEMO_POINTS=[
 dict(id='demo-a',label='Punto A · Madrid',lat=40.4331538,lon=-3.7040872,pano_id='4XV5XZ4Vo7SQAULn992JrQ',heading=195.89313310418282,pitch=3.8884,cohort='unverified',coordinates_reviewed=False),
 dict(id='demo-b',label='Punto B · Madrid',lat=40.4328741,lon=-3.704739,pano_id='VoK_pFOJdaEOzFov2O1FUg',heading=0,pitch=0,cohort='unverified',coordinates_reviewed=False),
]

def load_accidents(db_path,year_start=2019,year_end=2025,exclude_years=()):
    path=Path(db_path)
    if not path.is_file():raise FileNotFoundError(f'No existe la base de datos: {path}. Ejecuta el cuaderno en el ordenador que contiene el archivo.')
    with duckdb.connect(str(path),read_only=True) as con:
        columns=set(con.execute('DESCRIBE accidentes_todos').fetchdf()['column_name'])
        required={'accident_uid','year','con_peaton','apto_mapa','geolocation_quality','x_final','y_final','lat','lon'}
        if required-columns:raise ValueError(f'Faltan columnas: {sorted(required-columns)}')
        # Una fila por accidente, antes de cualquier cálculo espacial.
        df=con.execute('''SELECT accident_uid, year, location_raw, distrito,
          x_final, y_final, lat, lon, geolocation_quality
          FROM accidentes_todos WHERE con_peaton=TRUE AND apto_mapa=TRUE
          AND geolocation_quality IN ('high','medium') AND year BETWEEN ? AND ?
          AND x_final IS NOT NULL AND y_final IS NOT NULL''',[year_start,year_end]).fetchdf()
        coverage=con.execute('''SELECT year,COUNT(DISTINCT accident_uid) total_accidents,
          COUNT(DISTINCT CASE WHEN con_peaton THEN accident_uid END) pedestrian_accidents,
          COUNT(DISTINCT CASE WHEN con_peaton AND apto_mapa AND geolocation_quality IN ('high','medium') THEN accident_uid END) mapped_pedestrian_accidents
          FROM accidentes_todos GROUP BY year ORDER BY year''').fetchdf()
    if df.accident_uid.duplicated().any():raise ValueError('accidentes_todos contiene accident_uid repetidos: revísalos antes del análisis.')
    if exclude_years:df=df[~df.year.isin(exclude_years)].copy()
    finite=np.isfinite(df[['x_final','y_final']].to_numpy(dtype=float)).all(axis=1)
    df=df.loc[finite].reset_index(drop=True)
    if df.empty:raise ValueError('No hay accidentes aptos para el mapa en el periodo seleccionado.')
    coverage['mapped_pct']=100*coverage.mapped_pedestrian_accidents/coverage.pedestrian_accidents.replace(0,np.nan)
    return df,coverage

def dataset_version(df,year_start,year_end,radius_m,exclude_years=()):
    digest=hashlib.sha256()
    digest.update(json.dumps([year_start,year_end,radius_m,sorted(exclude_years),DEFINITION,QUALITY]).encode())
    # Incluye el contenido espacial, no solo el nombre del archivo o la fecha.
    stable=df[['accident_uid','x_final','y_final']].sort_values('accident_uid')
    digest.update(pd.util.hash_pandas_object(stable,index=False).to_numpy().tobytes())
    return digest.hexdigest()[:24]

def point_metrics(df,points,radius_m=25,year_start=2019,year_end=2025,exclude_years=()):
    if radius_m<=0:raise ValueError('El radio debe ser positivo')
    if not len(points):raise ValueError('Añade al menos una ubicación')
    xy=df[['x_final','y_final']].to_numpy(dtype=float)
    tree=cKDTree(xy)
    version=dataset_version(df,year_start,year_end,radius_m,exclude_years)
    output=[]
    for point in points:
        p=dict(point)
        if not (-4.2<=float(p['lon'])<=-3 and 40<=float(p['lat'])<=41):raise ValueError(f'Coordenadas fuera del entorno de Madrid: {p["id"]}')
        x,y=TO_UTM.transform(float(p['lon']),float(p['lat']))
        idx=tree.query_ball_point([x,y],radius_m)
        sample=df.iloc[idx]
        metric=dict(accidents=int(sample.accident_uid.nunique()),radius_m=float(radius_m),year_start=int(year_start),year_end=int(year_end),version=version,definition=DEFINITION,quality=QUALITY,verified=str(p.get('coordinates_reviewed',False)).strip().lower()=='true',excluded_years=list(exclude_years))
        p.update(x_utm=x,y_utm=y,metrics=metric)
        output.append(p)
    return output

def hotspot_candidates(df,radius_m=25,n=100,min_distance_m=None):
    """Centros candidatos en coordenadas de accidentes y selección sin solapes.

    No es un ranking oficial de cruces: es una regla espacial explícita.
    Separación >=2R impide que el top 5 muestre cinco centros del mismo entorno.
    """
    min_distance_m=max(2*radius_m,min_distance_m or 0)
    xy=df[['x_final','y_final']].to_numpy(dtype=float)
    candidates=np.unique(xy,axis=0)
    tree=cKDTree(xy)
    counts=tree.query_ball_point(candidates,radius_m,return_length=True)
    order=np.lexsort((candidates[:,1],candidates[:,0],-counts))
    selected=[];records=[]
    for k in order:
        center=candidates[k]
        if selected and np.min(np.linalg.norm(np.asarray(selected)-center,axis=1))<min_distance_m:continue
        selected.append(center)
        lon,lat=TO_WGS.transform(*center)
        nearest=tree.query(center)[1]
        row=df.iloc[nearest]
        records.append(dict(id=f'hotspot-{len(records)+1:03d}',label=str(row.location_raw),distrito=str(row.distrito),lat=lat,lon=lon,x_utm=center[0],y_utm=center[1],accidents=int(counts[k]),radius_m=radius_m,pano_id='',heading=0,pitch=0,cohort='high',coordinates_reviewed=False))
        if len(records)>=n:break
    return pd.DataFrame(records)

def story_geojson(df,hotspots,radius_m,year_start,year_end,exclude_years=()):
    # Mapa de toda la muestra en celdas de 50 m; ranking por buffers independiente.
    metadata=dict(radius_m=radius_m,year_start=year_start,year_end=year_end,excluded_years=list(exclude_years),definition=DEFINITION,quality=QUALITY,version=dataset_version(df,year_start,year_end,radius_m,exclude_years),method='centers_at_accident_coordinates;greedy_separation_at_least_2R',source='Accidentes con intervención de Policía Municipal; accidentes_todos',mapped_accidents=int(len(df)),map_grid_m=50)
    features=[dict(type='Feature',geometry=dict(type='Point',coordinates=[float(r.lon),float(r.lat)]),properties=dict(id=r.id,label=r.label,accidents=int(r.accidents),distrito=r.distrito)) for r in hotspots.itertuples()]
    hotspot_features=features
    grid=df.copy()
    grid['gx']=np.floor(grid.x_final/50).astype(int)
    grid['gy']=np.floor(grid.y_final/50).astype(int)
    cells=grid.groupby(['gx','gy']).agg(accidents=('accident_uid','nunique'),distrito=('distrito','first')).reset_index()
    map_features=[]
    for c in cells.itertuples():
        lon,lat=TO_WGS.transform((c.gx+.5)*50,(c.gy+.5)*50)
        map_features.append(dict(type='Feature',geometry=dict(type='Point',coordinates=[float(lon),float(lat)]),properties=dict(label=f'Celda de 50 m · {c.distrito}',accidents=int(c.accidents))))
    assert sum(f['properties']['accidents'] for f in map_features)==len(df)
    return dict(type='FeatureCollection',metadata=metadata,features=map_features,hotspots=hotspot_features)

def export_catalog(points,story,path):
    ids=set();clean=[]
    for p in points:
        if p['id'] in ids:raise ValueError('IDs de ubicaciones repetidos')
        ids.add(p['id'])
        if not p.get('pano_id'):raise ValueError(f'Falta el panorama de {p["id"]}. Selecciónalo antes de incluirlo en la encuesta.')
        clean.append({k:p[k] for k in ('id','label','lat','lon','pano_id','heading','pitch','cohort','metrics')})
    payload=dict(points=clean,story=story)
    Path(path).write_text(json.dumps(payload,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    return payload

def write_analysis_tables(db_path,points,story):
    """Añade solo tablas pc_*; las tablas originales de accidentes no se alteran.

    Cerrar conexiones de lectura y el visor local antes de abrir en escritura.
    La operación es transaccional; repetir reemplaza únicamente las métricas
    de los IDs del catálogo actual y añade versiones históricas del mapa.
    """
    with duckdb.connect(str(db_path)) as con:
        con.execute('BEGIN TRANSACTION')
        try:
            con.execute('''CREATE TABLE IF NOT EXISTS pc_points (
              point_id VARCHAR PRIMARY KEY,label VARCHAR,lat DOUBLE,lon DOUBLE,
              x_utm DOUBLE,y_utm DOUBLE,pano_id VARCHAR,heading DOUBLE,pitch DOUBLE,
              cohort VARCHAR,coordinates_reviewed BOOLEAN,updated_at TIMESTAMP)''')
            con.execute('''CREATE TABLE IF NOT EXISTS pc_point_metrics (
              point_id VARCHAR,metric_version VARCHAR,year_start INTEGER,year_end INTEGER,
              radius_m DOUBLE,accidents INTEGER,definition VARCHAR,quality VARCHAR,
              verified BOOLEAN,excluded_years_json VARCHAR,computed_at TIMESTAMP,
              PRIMARY KEY(point_id,metric_version))''')
            con.execute('''CREATE TABLE IF NOT EXISTS pc_story (
              metric_version VARCHAR PRIMARY KEY,geojson VARCHAR,computed_at TIMESTAMP)''')
            for p in points:
                m=p['metrics']
                con.execute('''INSERT INTO pc_points VALUES (?,?,?,?,?,?,?,?,?,?,?,current_timestamp)
                  ON CONFLICT(point_id) DO UPDATE SET label=excluded.label,lat=excluded.lat,
                  lon=excluded.lon,x_utm=excluded.x_utm,y_utm=excluded.y_utm,pano_id=excluded.pano_id,
                  heading=excluded.heading,pitch=excluded.pitch,cohort=excluded.cohort,
                  coordinates_reviewed=excluded.coordinates_reviewed,updated_at=now()''',
                  [p['id'],p['label'],p['lat'],p['lon'],p['x_utm'],p['y_utm'],p['pano_id'],p['heading'],p['pitch'],p['cohort'],m['verified']])
                con.execute('''INSERT INTO pc_point_metrics VALUES (?,?,?,?,?,?,?,?,?,?,current_timestamp)
                  ON CONFLICT(point_id,metric_version) DO UPDATE SET accidents=excluded.accidents,
                  verified=excluded.verified,computed_at=now()''',
                  [p['id'],m['version'],m['year_start'],m['year_end'],m['radius_m'],m['accidents'],m['definition'],m['quality'],m['verified'],json.dumps(m.get('excluded_years',[]))])
            con.execute('''INSERT INTO pc_story VALUES (?,?,current_timestamp)
              ON CONFLICT(metric_version) DO UPDATE SET geojson=excluded.geojson,computed_at=now()''',
              [story['metadata']['version'],json.dumps(story,ensure_ascii=False,allow_nan=False)])
            con.execute('COMMIT')
        except Exception:
            con.execute('ROLLBACK');raise

def read_responses(path):
    payload=json.loads(Path(path).read_text(encoding='utf-8'))
    if payload.get('schema_version')!=1:raise ValueError('Versión de exportación no compatible')
    sessions=pd.DataFrame(payload.get('sessions',[]));trials=pd.DataFrame(payload.get('trials',[]))
    if sessions.empty or trials.empty:return sessions,pd.DataFrame()
    if sessions.id.duplicated().any() or trials.id.duplicated().any():raise ValueError('IDs duplicados en la exportación')
    rows=[]
    for t in trials.to_dict('records'):
        if not t.get('choice'):continue
        a=json.loads(t['left_snapshot']);b=json.loads(t['right_snapshot'])
        ma,mb=a.get('metrics'),b.get('metrics')
        comparable=bool(ma and mb and ma.get('verified',True) and mb.get('verified',True) and all(ma.get(k)==mb.get(k) for k in ('version','year_start','year_end','radius_m','definition','quality')) and ma['accidents']!=mb['accidents'])
        decisive=t['choice'] in ('left','right');selected=a if t['choice']=='left' else b if t['choice']=='right' else None
        other=b if t['choice']=='left' else a if t['choice']=='right' else None
        pair='|'.join(sorted([a['id'],b['id']]))
        rows.append(dict(**t,pair_key=pair,left_accidents=ma['accidents'] if ma else np.nan,right_accidents=mb['accidents'] if mb else np.nan,metric_version=ma['version'] if ma and mb and ma['version']==mb['version'] else None,selected_point=selected['id'] if selected else None,other_point=other['id'] if other else None,decisive=decisive,comparable=comparable,agreement=(selected['metrics']['accidents']<other['metrics']['accidents']) if comparable and decisive else np.nan,duration_seconds=(t['duration_ms'] or 0)/1000,reasons=json.loads(t.get('reasons_json') or '[]')))
    votes=pd.DataFrame(rows)
    if votes.empty:return sessions,votes
    votes=votes.merge(sessions.rename(columns={'id':'session_id'}),on='session_id',how='left',validate='many_to_one')
    votes['repeated_pair']=votes.duplicated(['session_id','pair_key'],keep=False)
    votes['very_fast_flag']=votes.duration_seconds<1.5
    complete=votes.groupby('session_id').size().eq(5)
    votes['complete_session']=votes.session_id.map(complete)
    return sessions,votes

def analytic_sample(votes,include_demo=False,complete_only=True):
    if votes.empty:return votes.copy()
    keep=pd.Series(True,index=votes.index)
    if not include_demo:keep &= votes.study_mode.eq('study')
    if complete_only:keep &= votes.complete_session
    return votes.loc[keep].copy()

def preference_table(votes,group_by=()):
    if votes.empty:return pd.DataFrame()
    rows=[]
    for t in votes.to_dict('records'):
        for side in ('left','right'):
            p=json.loads(t[side+'_snapshot']);m=p.get('metrics') or {}
            rows.append(dict(point_id=p['id'],label=p['label'],session_id=t['session_id'],metric_version=m.get('version','unverified'),accidents=m.get('accidents',np.nan),chosen=int(t['choice']==side),decisive=int(t['decisive']),tie=int(t['choice']=='tie'),skip=int(t['choice']=='skip'),**{g:t[g] for g in group_by}))
    df=pd.DataFrame(rows)
    keys=['point_id','label','metric_version',*group_by]
    out=df.groupby(keys,dropna=False).agg(appearances=('chosen','size'),choices=('chosen','sum'),decisive=('decisive','sum'),participants=('session_id','nunique'),ties=('tie','sum'),skips=('skip','sum'),accidents=('accidents','first')).reset_index()
    out['preference_pct']=100*out.choices/out.decisive.replace(0,np.nan)
    return out

def cluster_interval(votes,field='agreement',n_bootstrap=2000,seed=42):
    """Intervalo bootstrap por participación, no por voto individual."""
    if votes.empty:return dict(n_participants=0,n_votes=0,estimate=None,ci95=None)
    df=votes.loc[votes[field].notna(),['session_id',field]].copy();df[field]=df[field].astype(float)
    by=df.groupby('session_id')[field].agg(['sum','count'])
    if by.empty:return dict(n_participants=0,n_votes=0,estimate=None,ci95=None)
    estimate=float(by['sum'].sum()/by['count'].sum())
    if len(by)<2:return dict(n_participants=len(by),n_votes=int(by['count'].sum()),estimate=estimate,ci95=None)
    arr=by[['sum','count']].to_numpy();rng=np.random.default_rng(seed)
    # Lotes evitan asignar una matriz 2000 x todos los participantes.
    boot=[]
    for _ in range(n_bootstrap):
        sample=arr[rng.integers(0,len(arr),size=len(arr))]
        boot.append(sample[:,0].sum()/sample[:,1].sum())
    return dict(n_participants=len(by),n_votes=int(by['count'].sum()),estimate=estimate,ci95=np.quantile(boot,[.025,.975]).tolist())

def bradley_terry(votes):
    """Ranking exploratorio regularizado por componentes de comparación.

    No comparar scores entre componentes desconectados. No produce p-valores.
    La constante mide la preferencia por la escena izquierda.
    """
    from sklearn.linear_model import LogisticRegression
    if votes.empty:return pd.DataFrame(),dict(status='no_data')
    decisive=votes[votes.decisive].copy()
    if len(decisive)<20:return pd.DataFrame(),dict(status='insufficient_votes',minimum=20)
    adjacency=defaultdict(set)
    for t in decisive.itertuples():adjacency[t.left_id].add(t.right_id);adjacency[t.right_id].add(t.left_id)
    remaining=set(adjacency);components=[]
    while remaining:
        root=min(remaining);todo=[root];component=set()
        while todo:
            point=todo.pop()
            if point in component:continue
            component.add(point);todo.extend(adjacency[point]-component)
        components.append(sorted(component));remaining-=component
    output=[];diagnostics=[]
    for num,nodes in enumerate(components,1):
        d=decisive[decisive.left_id.isin(nodes)&decisive.right_id.isin(nodes)]
        y=d.choice.eq('left').astype(int).to_numpy()
        if len(np.unique(y))<2:diagnostics.append(dict(component=num,status='only_one_response_side'));continue
        lookup={p:i for i,p in enumerate(nodes)};X=np.zeros((len(d),len(nodes)))
        for row,t in enumerate(d.itertuples()):X[row,lookup[t.left_id]]=1;X[row,lookup[t.right_id]]=-1
        model=LogisticRegression(C=1.0,fit_intercept=True,max_iter=3000,solver='lbfgs').fit(X,y)
        scores=model.coef_[0];scores=scores-scores.mean()
        output.extend(dict(point_id=p,component=num,score=float(score),votes=int(len(d))) for p,score in zip(nodes,scores))
        diagnostics.append(dict(component=num,points=len(nodes),votes=len(d),left_bias_log_odds=float(model.intercept_[0])))
    table=pd.DataFrame(output)
    if not table.empty:table=table.sort_values(['component','score'],ascending=[True,False])
    return table,dict(status='exploratory',components=len(components),diagnostics=diagnostics)

def association_model(votes):
    """Asociación descriptiva, con error estándar agrupado por participación.
    Resultado: elige izquierda. Predictor: log(1+accidentes derecha)-log(1+accidentes izquierda).
    No mide causalidad ni riesgo: no dispone de exposición peatonal o tráfico.
    """
    import statsmodels.api as sm
    d=votes[votes.decisive & votes.comparable].copy() if not votes.empty else pd.DataFrame()
    if len(d)<100 or d.session_id.nunique()<20:return None,'Se requieren al menos 100 elecciones comparables y 20 participaciones para este modelo exploratorio.'
    x=np.log1p(d.right_accidents)-np.log1p(d.left_accidents)
    if x.nunique()<2 or d.choice.nunique()<2:return None,'No hay variación suficiente para ajustar el modelo.'
    X=sm.add_constant(pd.DataFrame({'delta_log_accidents':x}))
    fit=sm.GLM(d.choice.eq('left').astype(int),X,family=sm.families.Binomial()).fit(cov_type='cluster',cov_kwds={'groups':d.session_id})
    return fit,'Asociación descriptiva; revisar separación, balance y sesgos de muestreo.'

def import_responses_duckdb(db_path,payload):
    """Replica una exportación para análisis. Nunca guarda cookies o hashes de acceso."""
    with duckdb.connect(str(db_path)) as con:
        con.execute('BEGIN TRANSACTION')
        try:
            con.execute('CREATE TABLE IF NOT EXISTS pc_sessions (session_id VARCHAR PRIMARY KEY,age_range VARCHAR,gender VARCHAR,urban VARCHAR,city VARCHAR,study_mode VARCHAR,created_at VARCHAR,catalog_hash VARCHAR)')
            con.execute('''CREATE TABLE IF NOT EXISTS pc_trials (trial_id VARCHAR PRIMARY KEY,session_id VARCHAR,round INTEGER,left_id VARCHAR,right_id VARCHAR,left_snapshot VARCHAR,right_snapshot VARCHAR,choice VARCHAR,reasons_json VARCHAR,duration_ms BIGINT,view_json VARCHAR,answered_at VARCHAR)''')
            for s in payload.get('sessions',[]):
                con.execute('''INSERT INTO pc_sessions VALUES (?,?,?,?,?,?,?,?) ON CONFLICT(session_id) DO NOTHING''',[s['id'],s['age_range'],s['gender'],s['urban'],s.get('city'),s['study_mode'],s['created_at'],s['catalog_hash']])
            for t in payload.get('trials',[]):
                con.execute('''INSERT INTO pc_trials VALUES (?,?,?,?,?,?,?,?,?,?,?,?) ON CONFLICT(trial_id) DO UPDATE SET choice=excluded.choice,reasons_json=excluded.reasons_json,duration_ms=excluded.duration_ms,view_json=excluded.view_json,answered_at=excluded.answered_at''',[t[k] for k in ('id','session_id','round','left_id','right_id','left_snapshot','right_snapshot','choice','reasons_json','duration_ms','view_json','answered_at')])
            con.execute('COMMIT')
        except Exception:con.execute('ROLLBACK');raise


## 3. Parámetros
El periodo inicial es **2019–2025**, con años completos y un formato de origen comparable. Puedes cambiarlo para usar 2010–2025, vigilando la cobertura de geolocalización por año. No se mezclan datos parciales de 2026 con años completos.

El radio de 25 m es una decisión de análisis, no un umbral oficial de peligrosidad. Comprueba la sensibilidad a 10 y 50 m. Se cuenta **cada `accident_uid` una sola vez**, en `accidentes_todos`, con `con_peaton = true`, `apto_mapa = true` y calidad `high`/`medium`. No se une con `personas_todos` para contar accidentes.

La escritura opcional añade solo tablas con prefijo `pc_`. Conserva una copia de tu archivo antes de activarla y cierra otras conexiones que lo estén abriendo en escritura.

In [ ]:
DB_PATH = Path(r"C:\Users\khora\Downloads\accidentesMadrid\accidentes_madrid.duckdb")
YEAR_START, YEAR_END = 2019, 2025
RADIUS_M = 25
EXCLUDE_YEARS = []  # Por defecto incluye 2020; evalúa una variante excluyéndolo.
OUT_DIR = DB_PATH.parent / "percepcion_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
WRITE_TO_DUCKDB = False  # True añade pc_points, pc_point_metrics y pc_story.


## 4. Lectura y cobertura
El porcentaje de accidentes geolocalizados ayuda a detectar si un año o distrito parece tener menos accidentes por falta de ubicación. La ausencia en el mapa no equivale a ausencia en los registros.

In [ ]:
accidents, coverage = load_accidents(DB_PATH, YEAR_START, YEAR_END, EXCLUDE_YEARS)
display(coverage)
print(f"Accidentes con peatón aptos para este análisis: {len(accidents):,}")
display(accidents.head())
coverage.to_csv(OUT_DIR / "cobertura_por_anio.csv", index=False, encoding="utf-8-sig")


## 5. Las dos ubicaciones del piloto
Los identificadores de panorama proceden de los enlaces que proporcionaste. Las coordenadas que aparecen en una URL de Google Maps pueden ser el centro de la cámara o del mapa; **revisa que sean la posición del panorama** antes de interpretar la concordancia.

Mantén `coordinates_reviewed=False` mientras no hayas revisado cada posición. Se calcularán recuentos provisionales, pero la plataforma no los utilizará para puntuar concordancia. Después de revisar una posición puedes poner su valor en `True` y volver a exportar. Cuando esté disponible Street View, el registro de la respuesta también guarda la posición real devuelta por el visor.

La segunda URL tenía un ángulo vertical hacia el cielo: aquí se inicia con `pitch=0` para mostrar el entorno peatonal. Mantén las condiciones de encuadre, zoom y fecha de imagen lo más comparables posible.

In [ ]:
POINTS = [dict(p) for p in DEMO_POINTS]
# Tras comprobar las posiciones en Street View:
# POINTS[0]["coordinates_reviewed"] = True
# POINTS[1]["coordinates_reviewed"] = True

# Para usar tu futuro catálogo de 100 puntos y 20–30 controles:
# data = pd.read_csv(OUT_DIR / "ubicaciones_estudio_revisadas.csv").to_dict("records")
# POINTS = data
# Columnas: id,label,lat,lon,pano_id,heading,pitch,cohort,coordinates_reviewed.
# cohort: high / control / unverified; no inferir seguridad a partir de la etiqueta.

measured_points = point_metrics(accidents, POINTS, RADIUS_M, YEAR_START, YEAR_END, EXCLUDE_YEARS)
point_table = pd.DataFrame([dict(id=p["id"], label=p["label"], lat=p["lat"], lon=p["lon"], **p["metrics"]) for p in measured_points])
display(point_table)


## 6. Sensibilidad al radio y al año 2020
Si la conclusión cambia al usar 10, 25 o 50 m, interprétala como dependiente del área elegida. Un buffer de 5 m puede resultar demasiado preciso para ubicaciones aproximadas de portales o cruces: úsalo solo cuando la precisión de ambos puntos lo justifique.

In [ ]:
sensitivity = []
for radius in [10, 25, 50]:
    points_r = point_metrics(accidents, POINTS, radius, YEAR_START, YEAR_END, EXCLUDE_YEARS)
    for p in points_r:
        sensitivity.append(dict(point_id=p["id"], radio_m=radius, accidentes=p["metrics"]["accidents"]))
display(pd.DataFrame(sensitivity).pivot(index="point_id", columns="radio_m", values="accidentes"))

without_2020 = accidents[accidents.year != 2020].copy()
if not without_2020.empty:
    variant = point_metrics(without_2020, POINTS, RADIUS_M, YEAR_START, YEAR_END, sorted(set(EXCLUDE_YEARS + [2020])))
    display(pd.DataFrame([dict(id=p["id"], accidentes_sin_2020=p["metrics"]["accidents"]) for p in variant]))


## 7. Candidatos de mayor accidentalidad y mapa de Madrid
Los focos se calculan contando accidentes a 25 m de coordenadas donde hubo accidentes y seleccionando centros separados al menos **2 × radio**. Así los cinco primeros no son cinco puntos del mismo cruce. Es una **regla espacial reproducible**, no un ranking oficial de cruces.

El mapa usa celdas de **50 m** y conserva todos los accidentes geolocalizados del periodo: la suma de sus recuentos debe coincidir con la muestra. Los cinco focos se mantienen separados del mapa agregado y se calculan con el radio configurado.

Los 100 candidatos necesitan un panorama y una revisión de encuadre antes de entrar en la encuesta. Los controles deben elegirse entre calles y cruces transitables comparables en distrito, tipología y exposición; no se generan controles en tejados o puntos aleatorios sin Street View.

In [ ]:
hotspots = hotspot_candidates(accidents, radius_m=RADIUS_M, n=100)
story = story_geojson(accidents, hotspots, RADIUS_M, YEAR_START, YEAR_END, EXCLUDE_YEARS)
display(hotspots[["id", "label", "distrito", "accidents", "lat", "lon"]].head(5))
print("Accidentes representados en el mapa:", sum(f["properties"]["accidents"] for f in story["features"]))
hotspots.to_csv(OUT_DIR / "candidatos_100_mayor_accidentalidad.csv", index=False, encoding="utf-8-sig")
columns = ["id", "label", "lat", "lon", "pano_id", "heading", "pitch", "cohort", "coordinates_reviewed"]
pd.DataFrame(columns=columns).to_csv(OUT_DIR / "plantilla_controles.csv", index=False, encoding="utf-8-sig")
(OUT_DIR / "mapa_accidentes.geojson").write_text(json.dumps(story, ensure_ascii=False, indent=2), encoding="utf-8")


## 8. Exportar el catálogo para la plataforma
Carga **`catalogo_estudio.json`** en la vista privada, sección «Incorporar la base de accidentes». El archivo contiene las ubicaciones, los recuentos, el periodo y el mapa; no sube el archivo DuckDB ni datos individuales de víctimas.

Con dos ubicaciones se mantiene el piloto de la pareja repetida. Con al menos cinco puntos `high` y cinco `control`, la plataforma crea cinco parejas distintas por participante, sin repetir una ubicación en su sesión y con orden izquierda/derecha aleatorio. Tu catálogo puede contener 100 y 20–30, respectivamente.

Cada sesión conserva una copia de las métricas y los panoramas con los que empezó. Actualizar el catálogo no reescribe respuestas anteriores.

In [ ]:
payload = export_catalog(measured_points, story, OUT_DIR / "catalogo_estudio.json")
print("Archivo listo:", OUT_DIR / "catalogo_estudio.json")
print("Verificación de coordenadas:", {p["id"]: p["metrics"]["verified"] for p in payload["points"]})


## 9. Añadir las tablas analíticas a tu DuckDB — opcional
Activa `WRITE_TO_DUCKDB=True` en la celda de parámetros y ejecuta esta celda. Las funciones SQL están incluidas arriba, por lo que puedes revisar el código antes de ejecutarlo.

| Tabla | Contenido |
|---|---|
| `pc_points` | Catálogo y panoramas de ubicaciones |
| `pc_point_metrics` | Recuentos, radio, periodo y versión de cálculo |
| `pc_story` | Mapa agregado y focos espaciales |
| `pc_sessions`, `pc_trials` | Se crean al importar las respuestas en la sección 12 |

`accidentes_todos`, `accidentes`, `personas_todos` y sus vistas no se modifican. Los votos de personas concurrentes se guardan en la base de la plataforma, no directamente en este archivo DuckDB.

In [ ]:
if WRITE_TO_DUCKDB:
    write_analysis_tables(DB_PATH, measured_points, story)
    print("Tablas pc_points, pc_point_metrics y pc_story actualizadas.")
else:
    print("Escritura desactivada. La lectura y el catálogo JSON ya están disponibles.")


## 10. Cargar las respuestas
En la vista privada pulsa «Exportar todo · JSON» y guarda el archivo en la carpeta `percepcion_output`. La exportación no incluye cookies ni hashes de acceso.

La unidad principal de análisis es la **participación**. Un identificador aleatorio no demuestra que dos participaciones pertenezcan a dos personas diferentes. El piloto repetido se excluye del análisis principal. «Similares» y «No puedo valorar» se describen aparte y no se consideran elecciones de un lugar.

In [ ]:
RESPONSES_PATH = OUT_DIR / "respuestas-a-pie.json"  # Ajusta al nombre descargado.
sessions, votes = read_responses(RESPONSES_PATH)
print("Participaciones:", len(sessions), "· respuestas guardadas:", len(votes))
if not votes.empty:
    display(votes.groupby(["study_mode", "choice"]).size().rename("n").reset_index())
    print("Respuestas muy rápidas marcadas para revisar:", int(votes.very_fast_flag.sum()))
    print("No se eliminan automáticamente por su duración.")

sample = analytic_sample(votes, include_demo=False, complete_only=True)
print("Respuestas en el análisis principal:", len(sample))
if sample.empty:
    print("Todavía no hay participaciones completas del estudio con parejas distintas. Las pruebas del piloto están en votes.")


## 11. Preferencias, concordancia y diferencias entre perfiles
La pregunta es «¿dónde te sentirías más cómodo/a?». Por tanto, hay concordancia cuando el lugar elegido tiene **menos** accidentes registrados que el otro, con el mismo periodo, radio, definición y versión. Los empates de accidentalidad y los recuentos sin verificar no son evaluables.

La concordancia no es una nota de acierto. La accidentalidad observada no mide riesgo sin denominadores de exposición — peatones que cruzan, tráfico o kilómetros recorridos. Un punto sin accidentes registrados no es necesariamente seguro.

El intervalo de concordancia se calcula por **bootstrap de participaciones completas**, conservando juntas sus cinco respuestas. Esto evita tratar los cinco votos de una misma participación como independientes.

In [ ]:
preferences = preference_table(sample)
by_age = preference_table(sample, group_by=["age_range"])
by_gender = preference_table(sample, group_by=["gender"])
by_city = preference_table(sample, group_by=["city"])
by_urban_residence = preference_table(sample, group_by=["urban"])
if not preferences.empty:
    display(preferences)
    preferences.to_csv(OUT_DIR / "preferencias_por_punto.csv", index=False, encoding="utf-8-sig")
    by_age.to_csv(OUT_DIR / "preferencias_por_edad.csv", index=False, encoding="utf-8-sig")
    by_gender.to_csv(OUT_DIR / "preferencias_por_genero.csv", index=False, encoding="utf-8-sig")
    by_city.to_csv(OUT_DIR / "preferencias_por_ciudad.csv", index=False, encoding="utf-8-sig")
    by_urban_residence.to_csv(OUT_DIR / "preferencias_residencia_urbana.csv", index=False, encoding="utf-8-sig")

agreement = cluster_interval(sample, field="agreement")
print("Concordancia e intervalo por participación:", agreement)
if not sample.empty:
    side_sample = sample[sample.decisive].copy()
    side_sample["left_preference"] = side_sample.choice.eq("left")
    print("Preferencia por la izquierda, control de sesgo:", cluster_interval(side_sample, field="left_preference"))
    reason_counts = sample.explode("reasons").groupby("reasons").size().sort_values(ascending=False)
    display(reason_counts.rename("votos_que_mencionan_el_motivo"))
    # Un mismo voto puede mencionar varios motivos; sus porcentajes no suman 100 %.

# Vista descriptiva del piloto, separada del análisis principal:
if not votes.empty:
    display(preference_table(votes[votes.study_mode.eq("demo")]))


## 12. Importar respuestas a DuckDB — opcional
Esta réplica analítica permite cruzar perfiles y respuestas en Jupyter. No sustituye la base transaccional de la plataforma. Repetir la importación no duplica las participaciones o votos y actualiza respuestas que antes estaban pendientes.

In [ ]:
if WRITE_TO_DUCKDB:
    response_payload = json.loads(RESPONSES_PATH.read_text(encoding="utf-8"))
    import_responses_duckdb(DB_PATH, response_payload)
    with duckdb.connect(str(DB_PATH), read_only=True) as con:
        display(con.execute("SELECT study_mode, COUNT(*) participaciones FROM pc_sessions GROUP BY study_mode").fetchdf())
else:
    print("Importación DuckDB desactivada. El análisis de la exportación JSON ya funciona.")


## 13. Ranking de percepción y asociación con la accidentalidad
El porcentaje bruto de preferencia depende de los rivales con los que se comparó cada lugar. Un modelo **Bradley–Terry** aproxima un ranking teniendo en cuenta esas comparaciones. Aquí es exploratorio y regularizado: no produce p-valores ni convierte la percepción en riesgo.

La función identifica componentes desconectados del grafo. No se pueden comparar scores entre componentes. Con una pareja repetida no se obtiene un ranking útil de Madrid.

La regresión utiliza la diferencia del logaritmo de los recuentos entre las dos escenas. Agrupa los errores por participación. Requiere un mínimo operativo de 100 elecciones comparables y 20 participaciones; esos mínimos no garantizan potencia estadística ni representatividad.

In [ ]:
ranking, ranking_diagnostics = bradley_terry(sample)
print(ranking_diagnostics)
if not ranking.empty:
    display(ranking)
    ranking.to_csv(OUT_DIR / "ranking_percepcion.csv", index=False, encoding="utf-8-sig")
model, model_note = association_model(sample)
print(model_note)
if model is not None:
    print(model.summary())
    (OUT_DIR / "modelo_asociacion.txt").write_text(str(model.summary()), encoding="utf-8")


## 14. Lectura de resultados y recorrido narrativo
Propuesta para el final de la experiencia:

1. **Tu mirada:** cuáles fueron los lugares elegidos y cuáles te parecieron similares.
2. **El mapa completo:** accidentes con peatón de la muestra, agrupados en celdas de 50 m.
3. **Cinco focos:** misma ventana temporal y buffers de 25 m, sin solapamiento de centros cercanos.
4. **Tu percepción frente a los registros:** concordancia en las comparaciones evaluables; sin lenguaje de «acertar/fallar».
5. **La mirada colectiva:** lugares cuyo aspecto invita a caminar pese a una acumulación alta, o que generan incomodidad con pocos registros. Mostrar el número de participaciones y reservar los grupos pequeños para análisis interno.

Para estudiar las discrepancias puedes cruzar percepción alta/baja y accidentalidad alta/baja. Define antes el umbral de accidentalidad — por ejemplo, un cuartil de una muestra comparable — y no escojas el umbral después para producir una historia llamativa. Los controles necesitan condiciones de exposición semejantes. Para el diseño futuro conviene añadir comparaciones dentro de cada grupo y entre tipologías, no solo «alto contra bajo», para mejorar la conectividad y poder ordenar lugares.

No se pondera la gravedad: las codificaciones de lesividad de 2010–2018 y 2019–2026 deben armonizarse antes. Cuenta víctimas en `personas_todos` y accidentes en `accidentes_todos`. Comprueba sesgos de geolocalización por año y distrito, autoselección de participantes, fecha de Street View, encuadre, orden y familiaridad con el lugar. La ciudad de residencia no mide por sí sola familiaridad con cada calle.

### Fuentes técnicas
- [Concurrencia en DuckDB](https://duckdb.org/docs/current/connect/concurrency.html)
- [Controles de Street View](https://developers.google.com/maps/documentation/javascript/reference/street-view)
- [Configurar Google Maps JavaScript API](https://developers.google.com/maps/documentation/javascript/get-api-key)
- [SQLite WAL](https://sqlite.org/wal.html)
